<a href="https://colab.research.google.com/github/toobajaved/GraphBasedMovieRecommendation/blob/main/Baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Milestone 4: Baseline Models

This notebook implements two baseline recommender systems:

1. Popularity-based recommendation
2. Matrix Factorization (SVD) using Surprise

These baselines will later be compared against the LightGCN model.

In [ ]:
import pandas as pd
import numpy as np

from surprise import Dataset
from surprise import Reader
from surprise import SVD

In [ ]:
train_pos = pd.read_csv("data/processed/train_pos.csv")
val_candidates = pd.read_csv("data/processed/val_candidates.csv")

print("Train positives:", len(train_pos))
print("Validation candidates:", len(val_candidates))

train_pos.head()

In [ ]:
movie_popularity = train_pos.groupby("movieId").size()
movie_popularity = movie_popularity.sort_values(ascending=False)

print("Top popular movies:")
movie_popularity.head(10)

In [ ]:
def popularity_score(movie_id):
    return movie_popularity.get(movie_id, 0)

In [ ]:
val_candidates["popularity_score"] = val_candidates["movieId"].apply(popularity_score)
val_candidates.head()

In [ ]:
train_mf = train_pos.copy()
train_mf["rating"] = 1

In [ ]:
reader = Reader(rating_scale=(0, 1))

data = Dataset.load_from_df(
    train_mf[["userId", "movieId", "rating"]],
    reader
)

trainset = data.build_full_trainset()

In [ ]:
model = SVD()

model.fit(trainset)

print("Matrix Factorization training complete.")

In [ ]:
def mf_score(row):
    prediction = model.predict(row["userId"], row["movieId"])
    return prediction.est

In [ ]:
val_candidates["mf_score"] = val_candidates.apply(mf_score, axis=1)

val_candidates.head()

In [ ]:
example_user = val_candidates["userId"].iloc[0]

user_movies = val_candidates[val_candidates["userId"] == example_user]

print("Ranking using Popularity:")
print(user_movies.sort_values("popularity_score", ascending=False).head(10))

print("\nRanking using Matrix Factorization:")
print(user_movies.sort_values("mf_score", ascending=False).head(10))

In [ ]:
val_candidates.to_csv(
    "data/processed/val_candidates_scored.csv",
    index=False
)

print("Scored validation candidates saved.")